<div style="background:linear-gradient(135deg,#7a3d00 0%,#b45309 55%,#d97706 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#ffe9c7;font-weight:700;text-transform:uppercase">Chapter 18 · Solutions</div>
  <div style="font-size:36px;font-weight:900;line-height:1.1;margin:10px 0 6px">Practice Challenges, Worked Answers ✅</div>
  <div style="font-size:15px;color:#ffe6cc;max-width:700px;line-height:1.6">Full solutions to the five "Data-Cleaning Mindset" challenges. Try them yourself first, then compare.</div>
  <div style="margin-top:16px;font-size:13px;color:#ffe2bf">Statistics, Data Science and AI: A Visual Handbook · John Fisher · 2026</div>
</div>

## ⚙️ Setup

In [1]:
import numpy as np
import pandas as pd
print("Ready.")

Ready.


<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#0891b2;letter-spacing:1px">CHALLENGE 1 · WRONG TYPES</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Spot the misread columns</div>
<div style="color:#4a5578;margin-top:6px">This frame has prices and quantities that won't do math. Use info()/dtypes to find which columns are stored as text instead of numbers, and explain why that happens on import.</div>
</div>

In [2]:
df = pd.DataFrame({"item":["A","B","C"], "price":["1,200","350","2,000"], "qty":["10","4","7"]})
print(df.dtypes)
print("\nTrying price.mean():")
try:
    print(df["price"].mean())
except Exception as e:
    print("  ERROR:", type(e).__name__)

item     str
price    str
qty      str
dtype: object

Trying price.mean():
  ERROR: TypeError


**Answer:** Both **price** and **qty** are `object` (text), not numbers. `price` carries thousands **commas** (`"1,200"`), so a CSV reader keeps it as a string; `qty` may have been read as text for similar formatting reasons. Numbers stored as text silently break arithmetic and sorting. The fix (next chapters) is to strip the commas and convert with `pd.to_numeric`, but first you have to **notice** via `dtypes`.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#7c3aed;letter-spacing:1px">CHALLENGE 2 · HIDDEN MISSING</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">The sentinel isna() misses</div>
<div style="color:#4a5578;margin-top:6px">A temperature column uses -999 to mean "sensor offline." Count how many readings are really missing, and show how the naive mean is corrupted.</div>
</div>

In [3]:
temp = pd.Series([21.4, 22.0, -999, 20.8, -999, 23.1, 19.9])
print(f"isna() count        : {temp.isna().sum()}   (sees nothing!)")
print(f"hidden -999 count   : {(temp==-999).sum()}")
print(f"naive mean          : {temp.mean():.1f}   <- corrupted")
print(f"mean ignoring -999  : {temp[temp!=-999].mean():.1f}")

isna() count        : 0   (sees nothing!)
hidden -999 count   : 2
naive mean          : -270.1   <- corrupted
mean ignoring -999  : 21.4


**Answer:** `isna()` finds **0** missing because -999 is a valid float. Really there are **2** missing readings. The naive mean (about -273) is nonsense; excluding the sentinel gives a sensible ~21.4. Lesson: you must know the dataset's missing **codes** and search for them, blanks are not the only kind of missing.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#d97706;letter-spacing:1px">CHALLENGE 3 · CATEGORY TANGLE</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Count the real categories</div>
<div style="color:#4a5578;margin-top:6px">A country column mixes spellings and casing. Use value_counts() to surface them, then count how many TRUE categories there really are.</div>
</div>

In [4]:
country = pd.Series(["USA","usa","U.S.A.","United States","UK","uk","U.K.","USA"])
print(country.value_counts())
print(f"\ndistinct labels: {country.nunique()}  but real countries: 2 (USA and UK)")

USA              2
usa              1
U.S.A.           1
United States    1
UK               1
uk               1
U.K.             1
Name: count, dtype: int64

distinct labels: 7  but real countries: 2 (USA and UK)


**Answer:** `value_counts()` shows several labels (USA, usa, U.S.A., United States, UK, uk, U.K.) that collapse to just **2 real categories**. Inconsistent casing and punctuation fragment one country into many, which would split counts and break group-bys. Standardizing these is Chapter 19; the mindset here is to **surface** them first.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#059669;letter-spacing:1px">CHALLENGE 4 · DUPES & RANGES</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Two fast audits</div>
<div style="color:#4a5578;margin-top:6px">Find exact duplicate rows and any impossible ages (outside 0 to 120) in this small table.</div>
</div>

In [5]:
df = pd.DataFrame({
    "id":[1,2,2,4,5],
    "name":["Ann","Bo","Bo","Cy","Di"],
    "age":[31, 27, 27, 250, 44],
})
print(f"exact duplicate rows: {df.duplicated().sum()}")
bad = df[(df["age"]<0)|(df["age"]>120)]
print(f"impossible ages     : {list(bad['age'])}")

exact duplicate rows: 1
impossible ages     : [250]


**Answer:** There is **1** exact duplicate row (the second "Bo, 27"), and one **impossible age** of 250. `duplicated()` flags repeated rows; a simple range rule flags values that cannot be real. Whether to drop the duplicate or investigate the 250 is a judgment call for later, the audit just makes them visible.

<div style="background:#fef3e2;border-left:5px solid #d97706;border-radius:10px;padding:16px 20px;font-family:Inter,sans-serif">
<span style="font-size:13px;font-weight:800;color:#2563eb;letter-spacing:1px">CHALLENGE 5 · TIDY IT UP</span>
<div style="font-size:21px;font-weight:800;color:#1a2138;margin-top:4px">Reshape messy-wide to tidy-long</div>
<div style="color:#4a5578;margin-top:6px">This sales table has years as COLUMN HEADERS, which breaks Wickham's first tidy rule (column headers are values, not variable names). Identify the broken rule and reshape it to tidy form with pd.melt.</div>
</div>

In [6]:
wide = pd.DataFrame({"store":["North","South"], "2021":[120,90], "2022":[135,110], "2023":[150,95]})
print("MESSY (years are column headers):"); print(wide)
tidy = wide.melt(id_vars="store", var_name="year", value_name="sales")
print("\nTIDY (each variable a column, each observation a row):"); print(tidy)

MESSY (years are column headers):
   store  2021  2022  2023
0  North   120   135   150
1  South    90   110    95

TIDY (each variable a column, each observation a row):
   store  year  sales
0  North  2021    120
1  South  2021     90
2  North  2022    135
3  South  2022    110
4  North  2023    150
5  South  2023     95


**Answer:** The years 2021, 2022, 2023 are **values, not variable names**, so the wide table breaks tidy rule 1 ("each variable forms a column"). `pd.melt` pulls them into a single **year** column with a matching **sales** column, so every row is now one observation (a store-year). Tidy data is about **shape**, not typos, and it is what lets grouping, plotting, and modeling just work.

---
<div style="background:#ffffff;border:1px solid #e6e9f2;border-radius:16px;padding:22px 26px;font-family:Inter,sans-serif">
<div style="font-size:19px;font-weight:800;color:#1a2138">🎉 Nicely done!</div>
<div style="color:#4a5578;line-height:1.8;margin-top:8px">You caught misread types, unmasked a hidden missing code, collapsed a category tangle, flagged duplicates and impossible values, and reshaped messy-wide data into tidy-long form. That is the cleaning mindset: inspect first, never trust data you have not audited.</div>
</div>

<div style="text-align:center;color:#8b94b3;font-size:12px;margin-top:16px">Statistics, Data Science and AI: A Visual Handbook · © 2026 John Fisher</div>